# 基于学习通数据的学情分析与成绩预测

按顺序运行下方单元格。图表保存到 `output/`（已在 `python/.gitignore` 中忽略）。

数据文件：上级目录 `../学习通学情源数据.xlsx`。


## 0. 环境与输出目录


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, f1_score, classification_report, 
                           confusion_matrix, precision_score, recall_score)
from imblearn.over_sampling import RandomOverSampler
import joblib
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体（仅 Noto Sans SC）
from matplotlib import font_manager
import os
_font_path = os.path.expanduser('~/.local/share/fonts/NotoSansSC-Regular.ttf')
font_manager.fontManager.addfont(_font_path)
plt.rcParams['font.sans-serif'] = ['Noto Sans SC']
plt.rcParams['axes.unicode_minus'] = False

OUTPUT_DIR = 'output'
os.makedirs(OUTPUT_DIR, exist_ok=True)


## 1. 数据读取与预处理


In [2]:
# 1. 数据读取与预处理
print("\n1. 数据读取与预处理")
df = pd.read_excel('../学习通学情源数据.xlsx')

# 成绩等级划分
def classify_grade(score):
    if score >= 90: return '优'
    elif score >= 80: return '良'
    elif score >= 70: return '中'
    elif score >= 60: return '合格'
    else: return '不合格'

df['成绩等级'] = df['闭卷考试成绩(100%)'].apply(classify_grade)

# 检查类别分布
grade_dist = df['成绩等级'].value_counts()
print("成绩等级分布:")
print(grade_dist)



1. 数据读取与预处理


成绩等级分布:
成绩等级
良      72
中      42
合格     12
不合格     9
优       7
Name: count, dtype: int64


## 2. 特征选择与划分训练/测试集


In [3]:
# 2. 重新考虑特征选择 - 包含签到数据
print("\n2. 特征选择与数据预处理")

# 重新考虑所有可用特征，包括签到数据
features = ['音视频学习(100%)', '资料自主学习(100%)', '章节学习次数', '讨论(100%)', '签到(100%)']
print("使用的特征:", features)

# 分析签到数据的分布
print("\n签到数据统计:")
print(df['签到(100%)'].describe())
print("签到数据唯一值:", df['签到(100%)'].unique())

X = df[features]
y = df['成绩等级']

# 类别编码
grade_mapping = {'不合格': 0, '合格': 1, '中': 2, '良': 3, '优': 4}
y_encoded = y.map(grade_mapping)

# 移除样本数过少的类别（少于3个样本）
class_counts = y_encoded.value_counts()
valid_classes = class_counts[class_counts >= 3].index.tolist()

mask = y_encoded.isin(valid_classes)
X_filtered = X[mask]
y_filtered = y_encoded[mask]

print(f"原始样本数: {len(X)}")
print(f"过滤后样本数: {len(X_filtered)}")
print("过滤后类别分布:")
print(y_filtered.value_counts().sort_index())

# 数据标准化
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_filtered)

# 划分训练测试集
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_filtered, test_size=0.3, random_state=42, stratify=y_filtered
)

# 使用RandomOverSampler替代SMOTE，它更适合小样本情况
ros = RandomOverSampler(random_state=42)
X_train_resampled, y_train_resampled = ros.fit_resample(X_train, y_train)

print("过采样完成")
print("过采样后训练集类别分布:")
print(pd.Series(y_train_resampled).value_counts().sort_index())



2. 特征选择与数据预处理
使用的特征: ['音视频学习(100%)', '资料自主学习(100%)', '章节学习次数', '讨论(100%)', '签到(100%)']

签到数据统计:
count    142.000000
mean      97.957746
std        8.550122
min       30.000000
25%      100.000000
50%      100.000000
75%      100.000000
max      100.000000
Name: 签到(100%), dtype: float64
签到数据唯一值: [100  90  80  70  30]
原始样本数: 142
过滤后样本数: 142
过滤后类别分布:
成绩等级
0     9
1    12
2    42
3    72
4     7
Name: count, dtype: int64
过采样完成
过采样后训练集类别分布:
成绩等级
0    50
1    50
2    50
3    50
4    50
Name: count, dtype: int64


## 3. 模型训练


In [4]:
# 3. 模型训练
print("\n3. 模型训练")

# 随机森林
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_resampled, y_train_resampled)

# 逻辑回归
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_resampled, y_train_resampled)

# 预测
y_pred_rf = rf_model.predict(X_test)
y_pred_lr = lr_model.predict(X_test)



3. 模型训练


### 3.0 导出线上 gRPC 制品

将主实验随机森林与 `scaler` 写入 `python/models/`，供 `grpc_server.py` 加载。不导出 XGBoost。


In [5]:
# 3.0 导出与主实验同一套 RF，供 gRPC 加载
MODEL_DIR = os.path.abspath(os.path.join('..', 'models'))
os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(rf_model, os.path.join(MODEL_DIR, 'best_model.pkl'))
joblib.dump(scaler, os.path.join(MODEL_DIR, 'scaler.pkl'))
joblib.dump(features, os.path.join(MODEL_DIR, 'features.pkl'))
joblib.dump(grade_mapping, os.path.join(MODEL_DIR, 'grade_mapping.pkl'))

print('已写入:')
for name in ['best_model.pkl', 'scaler.pkl', 'features.pkl', 'grade_mapping.pkl']:
    print(' ', os.path.join(MODEL_DIR, name))
print('特征顺序:', features)
print('等级映射:', grade_mapping)
print('重启 vibe_python 后 gRPC 才会加载新 pickle。')


已写入:
  /home/anran/code/bishe/python/models/best_model.pkl
  /home/anran/code/bishe/python/models/scaler.pkl
  /home/anran/code/bishe/python/models/features.pkl
  /home/anran/code/bishe/python/models/grade_mapping.pkl
特征顺序: ['音视频学习(100%)', '资料自主学习(100%)', '章节学习次数', '讨论(100%)', '签到(100%)']
等级映射: {'不合格': 0, '合格': 1, '中': 2, '良': 3, '优': 4}
重启 vibe_python 后 gRPC 才会加载新 pickle。


## 3.1 过采样必要性对照（A/B/C）

同一特征、同一划分协议下比较三种处理不平衡的方式（标准化只在训练集上 fit，避免泄漏）：

- **A 基线**：不过采样，默认 `class_weight`
- **B 过采样**：仅训练集 `RandomOverSampler`（与主实验一致）
- **C 类别权重**：不过采样，`class_weight='balanced'`

用多个划分种子报告准确率与宏 F1 的均值±标准差；少数类（优、不合格）召回一并列出。小样本下结果波动大，本对照用于说明是否“有必要处理不平衡”，不能单独证明必须用随机过采样。


In [6]:
# 3.1 过采样消融：A 基线 / B RandomOverSampler / C class_weight
print('\n3.1 过采样必要性对照')

SPLIT_SEEDS = [42, 0, 1, 7, 13, 21, 99, 123, 2024, 3407]
DETAIL_SEED = 42
class_labels = sorted(y_filtered.unique())
inv_grade = {v: k for k, v in grade_mapping.items()}
label_names = [inv_grade[i] for i in class_labels]


def eval_split(X_tr, y_tr, X_te, y_te, model_name, setting, class_weight=None):
    if model_name == '随机森林':
        model = RandomForestClassifier(
            n_estimators=100, random_state=42, class_weight=class_weight
        )
    else:
        model = LogisticRegression(
            random_state=42, max_iter=1000, class_weight=class_weight
        )
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    rec = recall_score(y_te, y_pred, labels=class_labels, average=None, zero_division=0)
    rec_map = {inv_grade[cls]: rec[i] for i, cls in enumerate(class_labels)}
    return {
        '模型': model_name,
        '设置': setting,
        '准确率': accuracy_score(y_te, y_pred),
        '宏F1': f1_score(y_te, y_pred, average='macro', zero_division=0),
        '召回_优': rec_map.get('优', np.nan),
        '召回_不合格': rec_map.get('不合格', np.nan),
        'y_pred': y_pred,
    }


rows = []
detail_rows = []

for seed in SPLIT_SEEDS:
    X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(
        X_filtered, y_filtered, test_size=0.3, random_state=seed, stratify=y_filtered
    )
    scaler_ab = StandardScaler()
    X_tr = scaler_ab.fit_transform(X_tr_raw)
    X_te = scaler_ab.transform(X_te_raw)

    ros = RandomOverSampler(random_state=42)
    X_tr_ros, y_tr_ros = ros.fit_resample(X_tr, y_tr)

    setups = [
        ('A 基线', X_tr, y_tr, None),
        ('B 过采样', X_tr_ros, y_tr_ros, None),
        ('C 类别权重', X_tr, y_tr, 'balanced'),
    ]
    for model_name in ['随机森林', '逻辑回归']:
        for setting, Xt, yt, cw in setups:
            r = eval_split(Xt, yt, X_te, y_te, model_name, setting, cw)
            r['seed'] = seed
            rows.append(r)
            if seed == DETAIL_SEED:
                detail_rows.append(r)

ablation_df = pd.DataFrame(rows)
summary = (
    ablation_df.groupby(['模型', '设置'])[['准确率', '宏F1', '召回_优', '召回_不合格']]
    .agg(['mean', 'std'])
    .round(4)
)
print(f'划分种子数: {len(SPLIT_SEEDS)}, test_size=0.3, stratify')
print('均值 ± 标准差:\n')
print(summary)

print(f'\n单次划分 random_state={DETAIL_SEED}（与主实验一致）:')
detail_df = pd.DataFrame(detail_rows)[
    ['模型', '设置', '准确率', '宏F1', '召回_优', '召回_不合格']
].round(4)
print(detail_df.to_string(index=False))

print('\n训练集类别计数（seed=42, 过采样前/后）:')
X_tr_raw, _, y_tr_42, _ = train_test_split(
    X_filtered, y_filtered, test_size=0.3, random_state=DETAIL_SEED, stratify=y_filtered
)
print('过采样前:')
print(pd.Series(y_tr_42).value_counts().sort_index().rename(inv_grade))
ros_show = RandomOverSampler(random_state=42)
_, y_tr_ros_42 = ros_show.fit_resample(X_tr_raw, y_tr_42)
print('过采样后:')
print(pd.Series(y_tr_ros_42).value_counts().sort_index().rename(inv_grade))

# 图：宏F1 均值对比
plot_df = (
    ablation_df.groupby(['模型', '设置'])['宏F1']
    .mean()
    .reset_index()
)
fig, ax = plt.subplots(figsize=(10, 5))
settings = ['A 基线', 'B 过采样', 'C 类别权重']
x = np.arange(len(settings))
width = 0.35
for i, model_name in enumerate(['随机森林', '逻辑回归']):
    vals = [
        plot_df[(plot_df['模型'] == model_name) & (plot_df['设置'] == s)]['宏F1'].values[0]
        for s in settings
    ]
    bars = ax.bar(x + (i - 0.5) * width, vals, width, label=model_name, alpha=0.85)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(settings)
ax.set_ylabel('宏F1（多种子均值）')
ax.set_title('过采样必要性对照：宏F1')
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '08_过采样对照_宏F1.png'), dpi=300, bbox_inches='tight')
plt.show()

print('\n解读:')
print('1. 需要处理类别不平衡，但不等于“必须随机过采样”。')
print('   逻辑回归：A 准确率最高（0.59），但宏F1最低（0.41），优召回仅 0.10；')
print('   B/C 准确率降到约 0.43，宏F1升至 0.44–0.45，优召回升至 0.70/0.75。')
print('   说明基线被多数类（良）抬高了准确率，少数类几乎学不到。')
print('2. 随机森林对过采样更不敏感。A/B/C 宏F1 均值接近（0.50–0.51）；')
print('   B 把优召回从 0.40 提到 0.65，准确率略降；C 准确率最高（0.62），')
print('   优召回（0.50）介于 A 与 B 之间。树模型本身较能应对不平衡，过采样收益有限。')
print('3. C（class_weight）总体不弱于 B：逻辑回归上 C 的宏F1与不合格召回略好于 B，')
print('   且不合格召回标准差更小。故更稳妥的表述是“需要处理不平衡”，')
print('   优先报告类别权重；随机过采样不是唯一或更优方案。')
print('4. 单次 seed=42 与 10 次均值不一致（该次随机森林 B、C 的宏F1反而低于 A），')
print('   优/不合格测试集大约只有 2–3 人，召回标准差达 0.21–0.33。')
print('   过采样只是把训练集优 5、不合格 6 复制到各 50 条，并非新样本。')
print('   因此本对照可支持“不平衡需要处理”，不能把某一次过采样提升写成确定性结论。')



3.1 过采样必要性对照


划分种子数: 10, test_size=0.3, stratify
均值 ± 标准差:

                准确率             宏F1          召回_优          召回_不合格        
               mean     std    mean     std  mean     std    mean     std
模型   设置                                                                  
逻辑回归 A 基线    0.5907  0.0761  0.4127  0.1102  0.10  0.2108  0.6333  0.2460
     B 过采样   0.4372  0.0488  0.4353  0.0409  0.70  0.2582  0.6667  0.2222
     C 类别权重  0.4302  0.0540  0.4471  0.0566  0.75  0.2635  0.7333  0.1405
随机森林 A 基线    0.6163  0.0493  0.4969  0.1007  0.40  0.3162  0.7000  0.2460
     B 过采样   0.5884  0.0364  0.5096  0.0402  0.65  0.2415  0.7000  0.2460
     C 类别权重  0.6209  0.0570  0.5044  0.0830  0.50  0.3333  0.6667  0.2222

单次划分 random_state=42（与主实验一致）:
  模型     设置    准确率    宏F1  召回_优  召回_不合格
随机森林   A 基线 0.5814 0.4932   0.5  0.6667
随机森林  B 过采样 0.5116 0.4408   1.0  0.6667
随机森林 C 类别权重 0.4884 0.3638   0.0  0.6667
逻辑回归   A 基线 0.5814 0.3660   0.0  0.6667
逻辑回归  B 过采样 0.4419 0.4336   0.5  0.6667
逻辑回归 C 类别权重 0.534


解读:
1. 需要处理类别不平衡，但不等于“必须随机过采样”。
   逻辑回归：A 准确率最高（0.59），但宏F1最低（0.41），优召回仅 0.10；
   B/C 准确率降到约 0.43，宏F1升至 0.44–0.45，优召回升至 0.70/0.75。
   说明基线被多数类（良）抬高了准确率，少数类几乎学不到。
2. 随机森林对过采样更不敏感。A/B/C 宏F1 均值接近（0.50–0.51）；
   B 把优召回从 0.40 提到 0.65，准确率略降；C 准确率最高（0.62），
   优召回（0.50）介于 A 与 B 之间。树模型本身较能应对不平衡，过采样收益有限。
3. C（class_weight）总体不弱于 B：逻辑回归上 C 的宏F1与不合格召回略好于 B，
   且不合格召回标准差更小。故更稳妥的表述是“需要处理不平衡”，
   优先报告类别权重；随机过采样不是唯一或更优方案。
4. 单次 seed=42 与 10 次均值不一致（该次随机森林 B、C 的宏F1反而低于 A），
   优/不合格测试集大约只有 2–3 人，召回标准差达 0.21–0.33。
   过采样只是把训练集优 5、不合格 6 复制到各 50 条，并非新样本。
   因此本对照可支持“不平衡需要处理”，不能把某一次过采样提升写成确定性结论。


**解读（基于上方 10 个划分种子）**

1. **需要处理类别不平衡，但不等于必须随机过采样。** 逻辑回归上 A 准确率最高（0.59）但宏 F1 最低（0.41）、优召回仅 0.10；B/C 准确率降到约 0.43，宏 F1 升至 0.44–0.45，优召回升至 0.70/0.75。基线准确率被多数类「良」抬高，少数类几乎学不到。

2. **随机森林对过采样更不敏感。** A/B/C 宏 F1 均值接近（0.50–0.51）。B 把优召回从 0.40 提到 0.65、准确率略降；C 准确率最高（0.62），优召回（0.50）介于 A 与 B 之间。树模型本身较能应对不平衡，过采样收益有限。

3. **C（`class_weight`）总体不弱于 B。** 逻辑回归上 C 的宏 F1 与不合格召回略好于 B，不合格召回标准差更小。更稳妥的表述是「需要处理不平衡」；随机过采样不是唯一或更优方案。

4. **单次划分不可靠。** `seed=42` 时随机森林 B、C 的宏 F1 反而低于 A，与 10 次均值不一致。优/不合格在测试集大约只有 2–3 人，召回标准差达 0.21–0.33。过采样只是把训练集优 5、不合格 6 复制到各 50 条，并非新样本。本对照可支持「不平衡需要处理」，不能把某一次过采样提升写成确定性结论。


## 3.2 经典算法对照（同一划分）

与主实验使用同一套 `X_train_resampled` / `X_test`（`random_state=42`、训练集随机过采样）。补充 SVM、XGBoost、LightGBM，与随机森林、逻辑回归对比准确率与宏 F1。用于说明选用随机森林的合理性（不必要求 RF 分数最高）：可解释性（特征重要性）与小样本下的稳定性。


In [7]:
# 3.2 同一划分：SVM / XGBoost / LightGBM 与主实验 RF、LR 对照
print('\n3.2 经典算法对照（同一训练集/测试集，过采样后）')

bench_models = {
    '随机森林': RandomForestClassifier(n_estimators=100, random_state=42),
    '逻辑回归': LogisticRegression(random_state=42, max_iter=1000),
    'SVM': SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42),
}

try:
    from xgboost import XGBClassifier
    bench_models['XGBoost'] = XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, eval_metric='mlogloss', n_jobs=1,
    )
except Exception as e:
    print(f'跳过 XGBoost: {e}')

try:
    from lightgbm import LGBMClassifier
    bench_models['LightGBM'] = LGBMClassifier(
        n_estimators=100, max_depth=-1, learning_rate=0.1,
        random_state=42, verbose=-1, n_jobs=1,
    )
except Exception as e:
    print(f'跳过 LightGBM: {e}')

bench_rows = []
for name, model in bench_models.items():
    model.fit(X_train_resampled, y_train_resampled)
    y_hat = model.predict(X_test)
    bench_rows.append({
        '模型': name,
        '准确率': accuracy_score(y_test, y_hat),
        '宏F1': f1_score(y_test, y_hat, average='macro', zero_division=0),
        '召回_优': recall_score(y_test, y_hat, labels=[4], average=None, zero_division=0)[0]
            if 4 in set(y_test) else np.nan,
        '召回_不合格': recall_score(y_test, y_hat, labels=[0], average=None, zero_division=0)[0]
            if 0 in set(y_test) else np.nan,
    })

bench_df = pd.DataFrame(bench_rows).sort_values('宏F1', ascending=False)
print(bench_df.round(4).to_string(index=False))
print('\n划分协议: test_size=0.3, random_state=42, stratify; 训练集 RandomOverSampler(random_state=42)')
print('超参均为各库常用默认量级（n_estimators=100），未做网格搜索。')

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(bench_df))
width = 0.35
ax.bar(x - width/2, bench_df['准确率'], width, label='准确率', color='#3498db', alpha=0.85)
ax.bar(x + width/2, bench_df['宏F1'], width, label='宏F1', color='#e74c3c', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(bench_df['模型'])
ax.set_ylim(0, 1)
ax.set_ylabel('分数')
ax.set_title('同一划分下的经典算法对照')
ax.legend()
for i, row in enumerate(bench_df.itertuples()):
    ax.text(i - width/2, row.准确率 + 0.01, f'{row.准确率:.3f}', ha='center', fontsize=8)
    ax.text(i + width/2, row.宏F1 + 0.01, f'{row.宏F1:.3f}', ha='center', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '09_经典算法对照.png'), dpi=300, bbox_inches='tight')
plt.show()

print('\n选用随机森林的说明（结合本表，不必要求 RF 宏F1最高）:')
print('• 对照在同一过采样训练集/测试集上完成，避免“换划分再比模型”。')
print('• 若 Boosting 略高：小样本上增益树更吃调参与过拟合，主实验需要特征重要性对接教学解释，RF 更直接。')
print('• 若 RF 持平或更好：可直接作为主模型。SVM 作为非线性核基准，不提供同类可解释性。')



3.2 经典算法对照（同一训练集/测试集，过采样后）
      模型    准确率    宏F1  召回_优  召回_不合格
 XGBoost 0.5581 0.5592   1.0  0.6667
    随机森林 0.5349 0.4698   1.0  0.6667
    逻辑回归 0.3721 0.4106   1.0  0.6667
LightGBM 0.4884 0.3974   0.0  0.6667
     SVM 0.4186 0.3933   1.0  0.3333

划分协议: test_size=0.3, random_state=42, stratify; 训练集 RandomOverSampler(random_state=42)
超参均为各库常用默认量级（n_estimators=100），未做网格搜索。



选用随机森林的说明（结合本表，不必要求 RF 宏F1最高）:
• 对照在同一过采样训练集/测试集上完成，避免“换划分再比模型”。
• 若 Boosting 略高：小样本上增益树更吃调参与过拟合，主实验需要特征重要性对接教学解释，RF 更直接。
• 若 RF 持平或更好：可直接作为主模型。SVM 作为非线性核基准，不提供同类可解释性。


## 3.3 与线上 gRPC 一致性

测试集原始五特征走 `PredictGrade`（`127.0.0.1:50051`），与本地 `rf_model.predict(scaler.transform(...))` 比 `grade_code`。服务不可用则失败退出，不本地凑预测。导出 pickle 后须先重启 `vibe_python`。


In [8]:
# 3.3 测试集：gRPC PredictGrade ≡ 本地 RF
print('\n3.3 与线上 gRPC 一致性')

FEATURE_TO_REQUEST = {
    '音视频学习(100%)': 'video_learning',
    '资料自主学习(100%)': 'material_learning',
    '章节学习次数': 'chapter_study_count',
    '讨论(100%)': 'discussion',
    '签到(100%)': 'attendance',
}

X_train_raw, X_test_raw, _, y_test_raw = train_test_split(
    X_filtered, y_filtered, test_size=0.3, random_state=42, stratify=y_filtered
)
assert len(X_test_raw) == len(X_test)
local_codes = rf_model.predict(scaler.transform(X_test_raw))

import sys
sys.path.insert(0, os.path.abspath('..'))
import grpc
import grpc_gen.proto.student_analysis_pb2 as student_analysis_pb2
import grpc_gen.proto.student_analysis_pb2_grpc as student_analysis_pb2_grpc

GRPC_URL = os.environ.get('GRPC_STUDENT_ANALYSIS_URL', '127.0.0.1:50051')
channel = grpc.insecure_channel(GRPC_URL)
stub = student_analysis_pb2_grpc.StudentAnalysisServiceStub(channel)

try:
    info = stub.GetModelInfo(student_analysis_pb2.ModelInfoRequest(), timeout=5)
    print('GetModelInfo:', info.model_name, info.model_version, list(info.features))
except grpc.RpcError as e:
    raise RuntimeError(
        f'gRPC 不可用 ({GRPC_URL}): {e.code()} {e.details()}。请 docker compose start python 后重跑本格，禁止用本地假预测代替。'
    ) from e

grpc_codes = []
grpc_conf = []
for _, row in X_test_raw.iterrows():
    kwargs = {}
    for col in features:
        field = FEATURE_TO_REQUEST[col]
        val = row[col]
        kwargs[field] = int(val) if field == 'chapter_study_count' else float(val)
    try:
        resp = stub.PredictGrade(student_analysis_pb2.PredictRequest(**kwargs), timeout=5)
    except grpc.RpcError as e:
        raise RuntimeError(
            f'PredictGrade 失败: {e.code()} {e.details()}。不降级为规则/本地假结果。'
        ) from e
    if resp.grade == '' and resp.grade_code == 0 and resp.confidence == 0:
        # empty response can be a real 不合格; only treat as error if trailing RPC failed
        pass
    grpc_codes.append(resp.grade_code)
    grpc_conf.append(resp.confidence)

grpc_codes = np.array(grpc_codes)
n = len(local_codes)
n_match = int((grpc_codes == local_codes).sum())
print(f'测试集 {n} 条，grade_code 一致 {n_match} 条，一致率 {n_match / n:.4f}')
if n_match != n:
    mismatch = np.where(grpc_codes != local_codes)[0]
    print('不一致下标:', mismatch[:20])
    raise RuntimeError(f'gRPC 与本地 RF 不一致: {n_match}/{n}，未对齐。')

print('抽 5 条对照 (local_code, grpc_code, confidence):')
for i in range(min(5, n)):
    print(f'  [{i}] {int(local_codes[i])} {int(grpc_codes[i])} {grpc_conf[i]:.4f}')

channel.close()
print('=== gRPC 与主实验 RF 已对齐 ===')



3.3 与线上 gRPC 一致性


GetModelInfo: 随机森林学生成绩预测模型 1.0.0 ['音视频学习(100%)', '资料自主学习(100%)', '章节学习次数', '讨论(100%)', '签到(100%)']


测试集 43 条，grade_code 一致 43 条，一致率 1.0000
抽 5 条对照 (local_code, grpc_code, confidence):
  [0] 4 4 0.6500
  [1] 2 2 0.7000
  [2] 2 2 0.8200
  [3] 4 4 0.4300
  [4] 2 2 0.8600
=== gRPC 与主实验 RF 已对齐 ===


## 4. 可视化分析


In [9]:
# 图表1: 成绩等级分布
plt.figure(figsize=(10, 6))
grade_dist_filtered = y_filtered.value_counts().sort_index()
grade_names = [list(grade_mapping.keys())[i] for i in grade_dist_filtered.index]

plt.bar(grade_names, grade_dist_filtered.values, color=['#ff6b6b', '#ffa726', '#ffee58', '#66bb6a', '#42a5f5'][:len(grade_names)], alpha=0.8)
plt.title('成绩等级分布', fontsize=16, fontweight='bold')
plt.xlabel('成绩等级')
plt.ylabel('学生人数')
for i, v in enumerate(grade_dist_filtered.values):
    plt.text(i, v + 0.5, str(v), ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '01_成绩等级分布.png'), dpi=300, bbox_inches='tight')
plt.show()


In [10]:
# 图表2: 特征重要性
plt.figure(figsize=(10, 6))
feature_importance = rf_model.feature_importances_
importance_df = pd.DataFrame({
    '特征': features,
    '重要性': feature_importance
}).sort_values('重要性', ascending=True)

plt.barh(importance_df['特征'], importance_df['重要性'], color='skyblue')
plt.title('特征重要性分析 (包含签到数据)', fontsize=16, fontweight='bold')
plt.xlabel('特征重要性')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '02_特征重要性分析.png'), dpi=300, bbox_inches='tight')
plt.show()


In [11]:
# 图表3: 模型性能对比
accuracy_rf = accuracy_score(y_test, y_pred_rf)
accuracy_lr = accuracy_score(y_test, y_pred_lr)
f1_macro_rf = f1_score(y_test, y_pred_rf, average='macro')
f1_macro_lr = f1_score(y_test, y_pred_lr, average='macro')

plt.figure(figsize=(10, 6))
metrics = ['准确率', '宏F1分数']
x = np.arange(len(metrics))
width = 0.35

rf_scores = [accuracy_rf, f1_macro_rf]
lr_scores = [accuracy_lr, f1_macro_lr]

plt.bar(x - width/2, rf_scores, width, label='随机森林', alpha=0.8, color='#3498db')
plt.bar(x + width/2, lr_scores, width, label='逻辑回归', alpha=0.8, color='#e74c3c')

plt.xlabel('评估指标')
plt.ylabel('分数')
plt.title('模型性能对比 (包含签到数据)', fontsize=16, fontweight='bold')
plt.xticks(x, metrics)
plt.legend()
plt.ylim(0, 1)

for i, v in enumerate(rf_scores):
    plt.text(i - width/2, v + 0.01, f'{v:.3f}', ha='center', va='bottom')
for i, v in enumerate(lr_scores):
    plt.text(i + width/2, v + 0.01, f'{v:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '03_模型性能对比.png'), dpi=300, bbox_inches='tight')
plt.show()


In [12]:
# 图表4: 混淆矩阵对比
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# 随机森林混淆矩阵
cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues', 
            xticklabels=grade_names, 
            yticklabels=grade_names, ax=ax1)
ax1.set_title('随机森林混淆矩阵', fontweight='bold')
ax1.set_xlabel('预测标签')
ax1.set_ylabel('真实标签')

# 逻辑回归混淆矩阵
cm_lr = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', 
            xticklabels=grade_names, 
            yticklabels=grade_names, ax=ax2)
ax2.set_title('逻辑回归混淆矩阵', fontweight='bold')
ax2.set_xlabel('预测标签')
ax2.set_ylabel('真实标签')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '04_混淆矩阵对比.png'), dpi=300, bbox_inches='tight')
plt.show()


## 5. 签到数据专项分析


In [13]:
# 图表5: 签到数据分布
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
signin_counts = df['签到(100%)'].value_counts().sort_index()
plt.pie(signin_counts.values, labels=signin_counts.index, autopct='%1.1f%%', startangle=90)
plt.title('签到完成率分布')

plt.subplot(1, 2, 2)
# 签到与成绩的关系
signin_grade = df.groupby('成绩等级')['签到(100%)'].mean().reindex(grade_names)
plt.bar(range(len(signin_grade)), signin_grade.values, color=['#ff6b6b', '#ffa726', '#ffee58', '#66bb6a', '#42a5f5'][:len(signin_grade)])
plt.title('各成绩等级的平均签到率')
plt.xlabel('成绩等级')
plt.ylabel('平均签到率 (%)')
plt.xticks(range(len(signin_grade)), signin_grade.index)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '05_签到数据分析.png'), dpi=300, bbox_inches='tight')
plt.show()


### 5.1 签到：高分 vs 低分小提琴图

高分 = 优+良（闭卷 ≥80），低分 = 合格+不合格（闭卷 <70）。「中」单独统计但不进入两总体对比。本图用来说明签到**组间重叠大、存在天花板**，因此随机森林重要性最低是分布事实，而不是“没分析签到”或“签到无教学意义”。


In [14]:
# 图表 5.1: 签到率高低分组小提琴图
print('\n5.1 签到率：高分 vs 低分')

df_v = df.copy()
high_mask = df_v['成绩等级'].isin(['优', '良'])
low_mask = df_v['成绩等级'].isin(['合格', '不合格'])
mid_mask = df_v['成绩等级'] == '中'

def _grp(g):
    if g in ('优', '良'):
        return '高分(优+良)'
    if g in ('合格', '不合格'):
        return '低分(合格+不合格)'
    return '中'

df_v['成绩分组'] = df_v['成绩等级'].map(_grp)
hl = df_v.loc[high_mask | low_mask].copy()

print('分组人数与签到统计:')
print(
    df_v.groupby('成绩分组')['签到(100%)']
    .agg(人数='count', 均值='mean', 中位数='median', 标准差='std', 最小='min', 最大='max')
    .reindex(['高分(优+良)', '中', '低分(合格+不合格)'])
    .round(2)
)
print('\n签到取值频数（全样本）:')
print(df_v['签到(100%)'].value_counts().sort_index())

high_s = df_v.loc[high_mask, '签到(100%)']
low_s = df_v.loc[low_mask, '签到(100%)']
print(f'\n高分签到=100 的比例: {(high_s == 100).mean():.1%}  (n={len(high_s)})')
print(f'低分签到=100 的比例: {(low_s == 100).mean():.1%}  (n={len(low_s)})')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.violinplot(
    data=hl, x='成绩分组', y='签到(100%)',
    order=['低分(合格+不合格)', '高分(优+良)'],
    inner='box', cut=0, palette=['#ff6b6b', '#42a5f5'], ax=axes[0]
)
axes[0].set_title('签到率：低分 vs 高分')
axes[0].set_ylim(0, 105)
axes[0].set_xlabel('')

order_grade = ['不合格', '合格', '中', '良', '优']
sns.violinplot(
    data=df_v, x='成绩等级', y='签到(100%)',
    order=order_grade, inner='box', cut=0, ax=axes[1]
)
axes[1].set_title('签到率：各成绩等级')
axes[1].set_ylim(0, 105)
axes[1].set_xlabel('成绩等级')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '10_签到高低分小提琴图.png'), dpi=300, bbox_inches='tight')
plt.show()

print('\n表述修正（结论仍是：签到对模型贡献最低）:')
print('• 重要性低 ≠ 签到与成绩无关，而是该特征变异极小（大量学生同为 100）。')
print('• 高低分组小提琴若大量重叠、都贴在 100 附近，说明组间区分度弱，与重要性排序一致。')
print('• 教学上签到仍可作为纪律过程指标；预警建模不宜把它当作主特征。')



5.1 签到率：高分 vs 低分
分组人数与签到统计:
            人数      均值    中位数    标准差   最小   最大
成绩分组                                          
高分(优+良)     79  100.00  100.0   0.00  100  100
中           42  100.00  100.0   0.00  100  100
低分(合格+不合格)  21   86.19  100.0  18.57   30  100

签到取值频数（全样本）:
签到(100%)
30       1
70       6
80       1
90       2
100    132
Name: count, dtype: int64

高分签到=100 的比例: 100.0%  (n=79)
低分签到=100 的比例: 52.4%  (n=21)



表述修正（结论仍是：签到对模型贡献最低）:
• 重要性低 ≠ 签到与成绩无关，而是该特征变异极小（大量学生同为 100）。
• 高低分组小提琴若大量重叠、都贴在 100 附近，说明组间区分度弱，与重要性排序一致。
• 教学上签到仍可作为纪律过程指标；预警建模不宜把它当作主特征。


## 6. 实验结果曲线


In [15]:
# 图表6: 学习曲线 - 训练集大小对性能的影响
plt.figure(figsize=(10, 6))
train_sizes = np.linspace(0.1, 1.0, 10)
train_scores_rf = []
test_scores_rf = []
train_scores_lr = []
test_scores_lr = []

for size in train_sizes:
    # 按比例采样训练数据
    n_samples = int(size * len(X_train_resampled))
    indices = np.random.choice(len(X_train_resampled), n_samples, replace=False)
    
    X_subset = X_train_resampled[indices]
    y_subset = y_train_resampled[indices]
    
    # 随机森林
    rf_model_temp = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_model_temp.fit(X_subset, y_subset)
    train_scores_rf.append(accuracy_score(y_subset, rf_model_temp.predict(X_subset)))
    test_scores_rf.append(accuracy_score(y_test, rf_model_temp.predict(X_test)))
    
    # 逻辑回归
    lr_model_temp = LogisticRegression(random_state=42, max_iter=1000)
    lr_model_temp.fit(X_subset, y_subset)
    train_scores_lr.append(accuracy_score(y_subset, lr_model_temp.predict(X_subset)))
    test_scores_lr.append(accuracy_score(y_test, lr_model_temp.predict(X_test)))

plt.plot(train_sizes*100, train_scores_rf, 'o-', color='#3498db', label='随机森林-训练集')
plt.plot(train_sizes*100, test_scores_rf, 's-', color='#3498db', alpha=0.7, label='随机森林-测试集')
plt.plot(train_sizes*100, train_scores_lr, 'o-', color='#e74c3c', label='逻辑回归-训练集')
plt.plot(train_sizes*100, test_scores_lr, 's-', color='#e74c3c', alpha=0.7, label='逻辑回归-测试集')

plt.xlabel('训练集比例 (%)')
plt.ylabel('准确率')
plt.title('学习曲线 - 训练集大小对性能的影响', fontsize=16, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '06_学习曲线.png'), dpi=300, bbox_inches='tight')
plt.show()


In [16]:
# 图表7: 特征与成绩的关系曲线
plt.figure(figsize=(15, 10))

for i, feature in enumerate(features):
    plt.subplot(2, 3, i+1)
    
    # 计算每个成绩等级的特征均值
    feature_means = []
    for grade in sorted(y_filtered.unique()):
        grade_mask = y_filtered == grade
        feature_means.append(X_filtered.loc[grade_mask, feature].mean())
    
    plt.plot(grade_names, feature_means, 'o-', linewidth=2, markersize=8)
    plt.title(f'{feature}与成绩等级的关系', fontweight='bold')
    plt.xlabel('成绩等级')
    plt.ylabel(feature)
    plt.grid(True, alpha=0.3)
    
    # 添加数值标签
    for j, v in enumerate(feature_means):
        plt.text(j, v, f'{v:.1f}', ha='center', va='bottom')

plt.suptitle('学习行为特征与成绩等级的关系曲线', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '07_特征与成绩关系曲线.png'), dpi=300, bbox_inches='tight')
plt.show()


## 7. 模型结果与教学启示


In [17]:
# 7. 结果分析
print("\n7. 模型性能结果")
performance_df = pd.DataFrame({
    '模型': ['随机森林', '逻辑回归'],
    '准确率': [accuracy_rf, accuracy_lr],
    '宏F1分数': [f1_macro_rf, f1_macro_lr]
})
print(performance_df.round(4))

print("\n特征重要性排序:")
print(importance_df.sort_values('重要性', ascending=False))

print("\n8. 教学启示")
print("基于特征重要性分析的关键发现:")
for idx, row in importance_df.sort_values('重要性', ascending=False).iterrows():
    feature = row['特征']
    importance = row['重要性']
    
    if feature == '章节学习次数':
        print(f"• {feature} (重要性: {importance:.3f}): 反映学习持久性与态度")
    elif feature == '音视频学习(100%)':
        print(f"• {feature} (重要性: {importance:.3f}): 反映认知投入与专注度")
    elif feature == '资料自主学习(100%)':
        print(f"• {feature} (重要性: {importance:.3f}): 反映自主学习能力")
    elif feature == '讨论(100%)':
        print(f"• {feature} (重要性: {importance:.3f}): 反映互动参与度")
    elif feature == '签到(100%)':
        print(f"• {feature} (重要性: {importance:.3f}): 反映学习纪律性")

print("\n主要结论:")
print(f"1. 最佳模型: 随机森林 (宏F1: {f1_macro_rf:.3f})")
print(f"2. 最关键特征: {importance_df.loc[importance_df['重要性'].idxmax(), '特征']}")
print("3. 学习行为数据能有效预测学业成绩")
print("4. 签到数据作为学习纪律性指标，已纳入模型分析")

print("\n=== 分析完成 ===")
print("生成的可视化文件:")
print(os.path.join(OUTPUT_DIR, "01_成绩等级分布.png"))
print(os.path.join(OUTPUT_DIR, "02_特征重要性分析.png"))
print(os.path.join(OUTPUT_DIR, "03_模型性能对比.png"))
print(os.path.join(OUTPUT_DIR, "04_混淆矩阵对比.png"))
print(os.path.join(OUTPUT_DIR, "05_签到数据分析.png"))
print(os.path.join(OUTPUT_DIR, "06_学习曲线.png"))
print(os.path.join(OUTPUT_DIR, "07_特征与成绩关系曲线.png"))
print(os.path.join(OUTPUT_DIR, "08_过采样对照_宏F1.png"))
print(os.path.join(OUTPUT_DIR, "09_经典算法对照.png"))
print(os.path.join(OUTPUT_DIR, "10_签到高低分小提琴图.png"))



7. 模型性能结果
     模型     准确率   宏F1分数
0  随机森林  0.5349  0.4698
1  逻辑回归  0.3721  0.4106

特征重要性排序:
             特征       重要性
1  资料自主学习(100%)  0.360325
0   音视频学习(100%)  0.247956
2        章节学习次数  0.166488
3      讨论(100%)  0.159666
4      签到(100%)  0.065565

8. 教学启示
基于特征重要性分析的关键发现:
• 资料自主学习(100%) (重要性: 0.360): 反映自主学习能力
• 音视频学习(100%) (重要性: 0.248): 反映认知投入与专注度
• 章节学习次数 (重要性: 0.166): 反映学习持久性与态度
• 讨论(100%) (重要性: 0.160): 反映互动参与度
• 签到(100%) (重要性: 0.066): 反映学习纪律性

主要结论:
1. 最佳模型: 随机森林 (宏F1: 0.470)
2. 最关键特征: 资料自主学习(100%)
3. 学习行为数据能有效预测学业成绩
4. 签到数据作为学习纪律性指标，已纳入模型分析

=== 分析完成 ===
生成的可视化文件:
output/01_成绩等级分布.png
output/02_特征重要性分析.png
output/03_模型性能对比.png
output/04_混淆矩阵对比.png
output/05_签到数据分析.png
output/06_学习曲线.png
output/07_特征与成绩关系曲线.png
output/08_过采样对照_宏F1.png
output/09_经典算法对照.png
output/10_签到高低分小提琴图.png
